# Stand up a Microsoft Foundry project and ship a first chat app on GPT-4o-mini

You just finished the AI-102 to AI-103 transition guide and the Foundry portal opens to a screen with three creation buttons labelled hub, project, and deployment, with no hint about the order or relationship. The previous engineer on this account standardised on key-in-environment-variable auth which security flagged last sprint. You have one session to stand up a clean hub plus project in eastus, deploy GPT-4o-mini at the new-account 30k TPM default, get a five-turn chat working over keyless auth, and read the usage metadata so the next teammate can see the real per-call cost before the team scales the deployment.

What you will build:

- A resource group, a Foundry hub, and a Foundry project under it in `eastus`.
- A `gpt-4o-mini` Standard deployment with 30k TPM (`sku.capacity=30`).
- An `AIProjectClient` authenticated via `DefaultAzureCredential` against the project endpoint (no API keys).
- A five-turn chat completion loop that reads `usage.prompt_tokens` and `usage.completion_tokens` from every response.

**Time.** Plan for about 45 minutes hands-on. Hub plus project provisioning can take 5 to 8 minutes on the Azure control plane; budget up to 75 minutes if the deployment quota call has to be retried. The cleanup cell at the bottom tears every resource down so the bill stops the moment you finish.

**Cost.** This lab costs less than a sip of coffee if you follow the steps and clean up. GPT-4o-mini is $0.15 per 1M input tokens and $0.60 per 1M output tokens. A five-turn debug loop with three retries still fits in under five cents. The Foundry hub and project orchestration layer is free at zero traffic.

This lab maps to AI-103 Domain 1: Plan and manage an Azure AI solution (28% of exam weight) and Domain 2: Implement generative AI and agentic solutions (33%).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the Azure SDKs this lab needs. Versions are pinned
# per LAB-CREATION-BLUEPRINT.md build rules.
# - azure-ai-projects 2.0 (March 2026) absorbed azure-ai-agents; never install
#   azure-ai-agents directly per the cert YAML risks list.
# - azure-identity 1.15 is the floor for DefaultAzureCredential's current
#   credential chain behavior.

!pip install --quiet labrun-checks[azure]==0.7.0 azure-identity>=1.15 azure-ai-projects==2.0.0 azure-mgmt-resource>=23.0.0 azure-mgmt-machinelearningservices>=1.0.0 azure-mgmt-cognitiveservices>=13.5.0 azure-mgmt-resourcegraph>=8.0.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import getpass
import json
import time

from azure.identity import DefaultAzureCredential
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError, ClientAuthenticationError
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.machinelearningservices import MachineLearningServicesMgmtClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest
from azure.ai.projects import AIProjectClient

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)
from labrun_checks.adapters.azure import AzureCleanupAdapter

LAB_ID = "foundry-project-and-chat-app"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "eastus"  # Foundry hosted agents GA region per cert YAML risks list

RESOURCE_GROUP = f"labrun-{LAB_ID}-rg"
HUB_NAME = f"labrun-{LAB_ID}-hub"
PROJECT_NAME = f"labrun-{LAB_ID}-project"
DEPLOYMENT_NAME = f"labrun-{LAB_ID}-gpt4omini"
MODEL_NAME = "gpt-4o-mini"
MODEL_VERSION = "2024-07-18"
MODEL_CAPACITY_TPM = 30  # sku.capacity unit is thousands of tokens-per-minute

# Resolved after DefaultAzureCredential validates.
SUBSCRIPTION_ID = None
AOAI_ACCOUNT_NAME = None
PROJECT_ENDPOINT = None
AZURE_CREDENTIAL = None

# Per-turn usage captured in Task 4. The wallet check reads from this.
USAGE_LOG = []

# GPT-4o-mini pricing as of 2026-05. Pricing claims in this notebook MUST match
# https://azure.microsoft.com/en-us/pricing/details/cognitive-services/openai-service/
PRICE_PER_1M_INPUT_USD = 0.15
PRICE_PER_1M_OUTPUT_USD = 0.60
COST_CEILING_USD = 0.02

In [ ]:
# NBVAL_SKIP
# Register the labrun session and validate Azure credentials via
# DefaultAzureCredential. The credential resolves environment variables first,
# then managed identity, then Azure CLI; in Colab the device-code path via
# `az login --use-device-code` is the usual entry point.
# This cell must succeed before the manifest cell creates anything per
# LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
subscription_id_input = getpass.getpass("AZURE_SUBSCRIPTION_ID: ").strip()
region_input = input(f"Azure region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"Foundry hosted agents are GA in eastus, northcentralus, swedencentral,")
    print(f"southeastasia, and japaneast. This lab pins {REGION}.")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

if not subscription_id_input:
    print("AZURE_SUBSCRIPTION_ID is required. Find yours with `az account show --query id -o tsv`.")
    raise SystemExit(1)

SUBSCRIPTION_ID = subscription_id_input

print("Asking DefaultAzureCredential to resolve your identity, hold on...")
try:
    AZURE_CREDENTIAL = DefaultAzureCredential()
    # Force a token fetch so the credential surfaces a clear error here, not
    # mid-lab after resources have been created.
    AZURE_CREDENTIAL.get_token("https://management.azure.com/.default")
except ClientAuthenticationError as e:
    print("DefaultAzureCredential could not resolve a credential.")
    print("In Colab, run `!az login --use-device-code` in a fresh cell and re-run setup.")
    print(f"  Inner error: {e}")
    raise SystemExit(1)

# Confirm the subscription is reachable. The list_locations call costs nothing
# and surfaces wrong-subscription errors before any resource creation.
resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
try:
    next(iter(resource_client.resource_groups.list()))
except HttpResponseError as e:
    print(f"Could not list resource groups in subscription {SUBSCRIPTION_ID}: {e.message}")
    print("Check that the subscription ID is correct and your principal has reader access.")
    raise SystemExit(1)
except StopIteration:
    pass  # An empty subscription is fine; we are about to populate it.

AZURE_CREDENTIALS_BAG = {
    "subscription_id": SUBSCRIPTION_ID,
    "region": REGION,
    "resource_group": RESOURCE_GROUP,
}

print(f"Credentials validated. Subscription: {SUBSCRIPTION_ID}")
print(f"Region: {REGION}")
print(f"Resource group target: {RESOURCE_GROUP}")
print("DefaultAzureCredential token fetched against management.azure.com.")

register(session_token=session_token, lab_id=LAB_ID, credentials=AZURE_CREDENTIALS_BAG)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan via Azure Resource
# Graph. Manifest is module-level and in reverse-creation order. This lab has
# no critical (hourly-billed) resources; Foundry orchestration and Standard
# GPT-4o-mini deployments do not bill at zero traffic. Per RESOURCE-SAFETY-SPEC
# Rule 4 the orphan scan BLOCKS execution if any prior tagged resources exist.
#
# The resource_group entry is the cross-cutting safety net for any Azure lab:
# deleting the resource group cascades through every resource underneath, so
# even if a specific adapter call fails the bill still stops.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="aoai_deployment",
        id=DEPLOYMENT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az cognitiveservices account deployment delete "
            f"--resource-group {RESOURCE_GROUP} "
            f"--name <attached-aoai-account> "
            f"--deployment-name {DEPLOYMENT_NAME}"
        ),
        extra={"account_name": AOAI_ACCOUNT_NAME},
    ),
    CleanupResource(
        type="foundry_project",
        id=PROJECT_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {PROJECT_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="foundry_hub",
        id=HUB_NAME,
        region=REGION,
        cli_delete_command=(
            f"az ml workspace delete --resource-group {RESOURCE_GROUP} "
            f"--name {HUB_NAME} --yes --no-wait"
        ),
    ),
    CleanupResource(
        type="resource_group",
        id=RESOURCE_GROUP,
        region=REGION,
        cli_delete_command=f"az group delete --name {RESOURCE_GROUP} --yes --no-wait",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Resource Graph query for any resource tagged labrun:lab-slug=<this lab>.
    Mirrors the AWS Resource Groups Tagging API path used by SAA-C03 labs.
    """
    graph_client = ResourceGraphClient(AZURE_CREDENTIAL)
    query = (
        f"Resources | where tags['{LAB_TAG_KEY}'] == '{LAB_TAG_VALUE}' "
        f"| project id, name, type, location"
    )
    request = QueryRequest(subscriptions=[SUBSCRIPTION_ID], query=query)
    response = graph_client.resources(request)
    return [row.get("id", "") for row in (response.data or [])]


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources or")
    print("a deployment that fails with QuotaExceeded.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print(f"manually: az group delete --name {RESOURCE_GROUP} --yes --no-wait")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the resource group and the Foundry hub plus project

Foundry's portal opens with three buttons labelled hub, project, and deployment, and the relationship is not obvious. A **hub** is the umbrella container with shared connections, identity, and storage; a **project** is the working unit with its own endpoint and deployments. You always create the hub first, then projects under it. A team with three apps building against the same models still creates three projects under one hub so each app gets a clean endpoint and its own deployment list, while the underlying compute, identity, and storage are pooled at the hub.

Build it in this order:

1. Create the resource group in `eastus` and tag it with `labrun:lab-slug=foundry-project-and-chat-app` at creation.
2. Call `MachineLearningServicesMgmtClient.workspaces.begin_create_or_update` to create the hub workspace (`kind=Hub`). Pass the tag explicitly in the `tags` field of the workspace payload. Wait for the long-running operation to complete.
3. Call the same method to create the project workspace (`kind=Project`) under the hub. The project must reference the hub via `hub_resource_id` in its properties. Tag it the same way.

Hub plus project provisioning is async. The hub create can take 4 to 6 minutes; the project create another 2 to 3 minutes. The cell prints a progress line while it waits.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the resource group, the Foundry hub, and the Foundry project.
# All three resources carry the lab tag so the orphan scan and Resource Graph
# queries can find them.

resource_client = ResourceManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

lab_tags = {LAB_TAG_KEY: LAB_TAG_VALUE}

# Resource group first. The atexit fallback will delete this on kernel exit if
# anything below fails, taking every child resource with it.
# YOUR CODE: Create the resource group using
# resource_client.resource_groups.create_or_update(RESOURCE_GROUP,
# {"location": REGION, "tags": lab_tags}).
print(f"Resource group created: {RESOURCE_GROUP}")

# Foundry hub. Workspaces with kind=Hub are the umbrella resource that owns
# the shared Azure OpenAI account, the storage account, and the identity
# scope used by every project under it.
hub_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Hub",
    "identity": {"type": "SystemAssigned"},
    "properties": {
        "friendly_name": HUB_NAME,
        "public_network_access": "Enabled",
    },
}
print("Asking ARM to allocate a Foundry hub, hold on for about 4 to 6 minutes...")
# YOUR CODE: Start the hub create via
# ml_client.workspaces.begin_create_or_update(RESOURCE_GROUP, HUB_NAME, hub_payload)
# and call .result() on the poller. Store the result in hub_workspace.

HUB_RESOURCE_ID = hub_workspace.id
print(f"Foundry hub provisioned: {HUB_NAME}")
print(f"Hub ARM id: {HUB_RESOURCE_ID}")

# Foundry project. Workspaces with kind=Project are the working unit each app
# uses. The hub_resource_id link is how the project inherits identity,
# storage, and the attached Azure OpenAI account from its hub.
project_payload = {
    "location": REGION,
    "tags": lab_tags,
    "kind": "Project",
    "identity": {"type": "SystemAssigned"},
    "properties": {
        "friendly_name": PROJECT_NAME,
        "hub_resource_id": HUB_RESOURCE_ID,
    },
}
print("Watching the project workspace go through provisioning, this takes 2 to 3 minutes...")
# YOUR CODE: Start the project create via
# ml_client.workspaces.begin_create_or_update(RESOURCE_GROUP, PROJECT_NAME, project_payload)
# and call .result() on the poller. Store the result in project_workspace.

PROJECT_ENDPOINT = project_workspace.properties.discovery_url
print(f"Foundry project provisioned: {PROJECT_NAME}")
print(f"Project endpoint: {PROJECT_ENDPOINT}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Hub and project workspaces exist in eastus with the lab tag,
# the hub is kind=Hub, the project is kind=Project, and the project's
# hub_resource_id resolves to the hub's full ARM id.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        ml = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

        try:
            hub = ml.workspaces.get(RESOURCE_GROUP, HUB_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=f"Foundry hub {HUB_NAME} does not exist in {RESOURCE_GROUP}.",
            )
        if (hub.kind or "").lower() != "hub":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Workspace {HUB_NAME} has kind={hub.kind!r}, expected 'Hub'. "
                    f"Hubs are the umbrella resource; projects sit underneath them."
                ),
            )
        if (hub.location or "").lower() != REGION:
            return CheckpointResult(
                status="fail",
                error_reason=f"Hub location is {hub.location!r}, expected {REGION!r}.",
            )
        hub_tags = hub.tags or {}
        if hub_tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Hub {HUB_NAME} is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found tags: {hub_tags}"
                ),
            )
        prov = getattr(hub.properties, "provisioning_state", None) if hub.properties else None
        if prov and prov != "Succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=f"Hub provisioningState is {prov!r}, expected 'Succeeded'.",
            )

        try:
            project = ml.workspaces.get(RESOURCE_GROUP, PROJECT_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=f"Foundry project {PROJECT_NAME} does not exist in {RESOURCE_GROUP}.",
            )
        if (project.kind or "").lower() != "project":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Workspace {PROJECT_NAME} has kind={project.kind!r}, expected 'Project'."
                ),
            )
        project_tags = project.tags or {}
        if project_tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Project {PROJECT_NAME} is missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Found tags: {project_tags}"
                ),
            )
        hub_ref = getattr(project.properties, "hub_resource_id", None) if project.properties else None
        if not hub_ref or hub.id not in hub_ref:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Project {PROJECT_NAME} hub_resource_id is {hub_ref!r}, expected to "
                    f"contain hub ARM id {hub.id!r}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint tells you exactly which resource is missing, has the wrong kind, or is missing the lab tag. Read the message before peeking further. A common shape: the hub provisioned fine but the project create returned before the LRO completed, so the project shows `provisioningState=Creating` instead of `Succeeded`.

</details>

<details><summary>Hint 2 (direction)</summary>

Two workspace creates, hub first. Each call returns an `LROPoller`; you must call `.result()` (or `.wait()`) before moving on, otherwise the project create races the hub provisioning. The project payload needs `hub_resource_id` pointing to the hub's full ARM id (the value returned in `hub_workspace.id` after the hub create completes). Both payloads need `kind` and `tags` set explicitly.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP,
    {"location": REGION, "tags": lab_tags},
)

hub_poller = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, HUB_NAME, hub_payload,
)
hub_workspace = hub_poller.result()

project_payload["properties"]["hub_resource_id"] = hub_workspace.id
project_poller = ml_client.workspaces.begin_create_or_update(
    RESOURCE_GROUP, PROJECT_NAME, project_payload,
)
project_workspace = project_poller.result()
```

</details>

**Wallet check.** Still at $0.00. The Foundry hub, project, and resource group are free at the orchestration layer. ARM control-plane calls are also free. The wallet does not move until you start generating tokens in Task 4. Your coffee this morning still wins by a wide margin.

## Task 2: Deploy GPT-4o-mini at 30k TPM Standard SKU

Foundry hubs provision an attached Azure OpenAI account behind the scenes. You will deploy the model into that attached account, not a fresh standalone account, which matches the path the Microsoft Learn AI-103T00-A course module 2 takes.

Build it in this order:

1. Find the Azure OpenAI account name attached to the hub. List the project's connections via the management plane and pick out the one whose `category` is `AzureOpenAI`. The connection's target points to the attached account.
2. Call `CognitiveServicesManagementClient.deployments.begin_create_or_update` against that account with `model.name=gpt-4o-mini`, `model.version=2024-07-18`, `sku.name=Standard`, `sku.capacity=30`. Wait for the LRO to complete.

**Why `Standard` not `Provisioned`.** Provisioned throughput is hourly-billed capacity charged whether you use it or not. This lab is pay-per-token; never opt into Provisioned for a workshop or a small workload. The validator rejects `Provisioned` or `GlobalProvisioned` to lock that lesson in.

**Why 30k TPM.** New Azure subscriptions get a 30k tokens-per-minute default quota on GPT-4o-mini in `eastus`. The lab pins `sku.capacity=30`. If your subscription has a lower quota (prior usage or region restrictions), the deployment call fails with `QuotaExceeded` and the hint points you at the quota request path.

In [ ]:
# NBVAL_SKIP
# Task 2: Discover the hub's attached Azure OpenAI account, then deploy
# GPT-4o-mini at 30k TPM Standard SKU into it.

ml_client = MachineLearningServicesMgmtClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
cs_client = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)

# Discover the attached AOAI account by listing connections on the hub.
# Hubs expose their attached AOAI account as a connection of category=AzureOpenAI.
AOAI_ACCOUNT_NAME = None
for conn in ml_client.connections.list(RESOURCE_GROUP, HUB_NAME):
    if (conn.properties.category or "").lower() == "azureopenai":
        # The connection target is the AOAI account's ARM resource path; the
        # account name is the trailing segment after /accounts/.
        target = conn.properties.target or ""
        if "/accounts/" in target:
            AOAI_ACCOUNT_NAME = target.split("/accounts/")[-1].split("/")[0]
            break
if not AOAI_ACCOUNT_NAME:
    print("Could not find an attached Azure OpenAI account on the hub.")
    print("Foundry hubs should provision one automatically; if this fails,")
    print("create one manually via the portal and re-run this cell.")
    raise SystemExit(1)
print(f"Attached Azure OpenAI account: {AOAI_ACCOUNT_NAME}")

deployment_payload = {
    "sku": {"name": "Standard", "capacity": MODEL_CAPACITY_TPM},
    "properties": {
        "model": {
            "format": "OpenAI",
            "name": MODEL_NAME,
            "version": MODEL_VERSION,
        },
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}

print("Watching the model deployment go through Succeeded purgatory, about 1 to 2 minutes...")
# YOUR CODE: Start the deployment create via
# cs_client.deployments.begin_create_or_update(RESOURCE_GROUP, AOAI_ACCOUNT_NAME,
# DEPLOYMENT_NAME, deployment_payload) and call .result() on the poller.
# Store the result in deployment_result.

print(f"GPT-4o-mini deployment provisioned: {DEPLOYMENT_NAME}")
print(f"  Model: {deployment_result.properties.model.name} v{deployment_result.properties.model.version}")
print(f"  SKU: {deployment_result.sku.name} capacity={deployment_result.sku.capacity}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Deployment exists with model=gpt-4o-mini, version=2024-07-18,
# sku.name=Standard, sku.capacity=30, provisioningState=Succeeded. Rejects
# Provisioned or GlobalProvisioned SKUs because they bill hourly regardless
# of traffic and this lab's cost model assumes pay-per-token.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        cs = CognitiveServicesManagementClient(AZURE_CREDENTIAL, SUBSCRIPTION_ID)
        if not AOAI_ACCOUNT_NAME:
            return CheckpointResult(
                status="fail",
                error_reason="AOAI_ACCOUNT_NAME is unset. Re-run Task 2 to discover the attached account.",
            )

        try:
            deployment = cs.deployments.get(RESOURCE_GROUP, AOAI_ACCOUNT_NAME, DEPLOYMENT_NAME)
        except ResourceNotFoundError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Deployment {DEPLOYMENT_NAME} does not exist on account {AOAI_ACCOUNT_NAME}."
                ),
            )

        model = deployment.properties.model
        if (model.name or "").lower() != MODEL_NAME:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Deployment model is {model.name!r}, expected {MODEL_NAME!r}. "
                    f"This lab pins GPT-4o-mini for the published $0.15/$0.60 pricing."
                ),
            )
        if model.version != MODEL_VERSION:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Model version is {model.version!r}, expected {MODEL_VERSION!r}. "
                    f"Pin the version explicitly so the lab is deterministic."
                ),
            )

        sku_name = (deployment.sku.name or "") if deployment.sku else ""
        if sku_name.lower() != "standard":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"SKU is {sku_name!r}, expected 'Standard'. Provisioned and "
                    f"GlobalProvisioned bill hourly even at zero traffic; this lab "
                    f"uses pay-per-token Standard."
                ),
            )
        capacity = deployment.sku.capacity if deployment.sku else 0
        if capacity != MODEL_CAPACITY_TPM:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"sku.capacity is {capacity}, expected {MODEL_CAPACITY_TPM} (30k TPM). "
                    f"The capacity unit is thousands of tokens-per-minute."
                ),
            )

        prov = deployment.properties.provisioning_state
        if prov != "Succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Deployment provisioningState is {prov!r}, expected 'Succeeded'. "
                    f"Did you call .result() on the begin_create_or_update poller?"
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint failure points at the field that is wrong: model name, version, SKU name, capacity, or provisioning state. If it says `QuotaExceeded` somewhere in the trace, your subscription's GPT-4o-mini quota in eastus is below 30k TPM; file a quota request from the Foundry portal Quotas tab.

</details>

<details><summary>Hint 2 (direction)</summary>

`sku.capacity=30` represents 30k tokens-per-minute; the value is the number of thousands, not the literal token count. `sku.name="Standard"`. Model `name="gpt-4o-mini"`, `version="2024-07-18"`, `format="OpenAI"`. The poller from `begin_create_or_update` is async; call `.result()` and inspect `deployment_result.properties.provisioning_state` for `"Succeeded"` before moving on.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
deployment_payload = {
    "sku": {"name": "Standard", "capacity": 30},
    "properties": {
        "model": {
            "format": "OpenAI",
            "name": "gpt-4o-mini",
            "version": "2024-07-18",
        },
        "version_upgrade_option": "OnceCurrentVersionExpired",
    },
}

poller = cs_client.deployments.begin_create_or_update(
    RESOURCE_GROUP, AOAI_ACCOUNT_NAME, DEPLOYMENT_NAME, deployment_payload,
)
deployment_result = poller.result()
```

</details>

**Wallet check.** Still at $0.00. Creating an Azure OpenAI deployment is a control-plane call; the bill does not move until you start firing chat completions. The deployment itself has no hourly fee at Standard SKU. Coffee continues to dominate the daily expense report.

## Task 3: Authenticate an AIProjectClient with DefaultAzureCredential

Production deployments require keyless auth. Your previous teammate left an API key in an environment variable and security flagged it last sprint. `DefaultAzureCredential` is the canonical way to authenticate to Foundry: it walks the credential chain (env vars, managed identity, Azure CLI, VS Code) and returns the first one that works. In containers it resolves the managed identity; in local development it resolves your `az login`. The application code is the same either way.

Build it in this order:

1. Construct `DefaultAzureCredential()`. Same credential type you already used to validate the subscription in setup.
2. Instantiate `AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cred)`. The endpoint comes from the project workspace's `discovery_url`.
3. Call one cheap project-level method (`client.connections.list()`) to prove the credential actually resolves against the project endpoint. Returning a non-error iterator is enough.

Trap to avoid: do NOT pass an `AzureKeyCredential` here. The checkpoint inspects the credential type and rejects keys. The Foundry account API key still exists in the portal; it is just not what production code should use.

In [ ]:
# NBVAL_SKIP
# Task 3: Construct the AIProjectClient against the project endpoint with
# DefaultAzureCredential. Run one cheap read to prove the credential resolved.

print("Pinging the project endpoint with the keyless credential...")

# YOUR CODE: Construct cred = DefaultAzureCredential() (you can reuse
# AZURE_CREDENTIAL from setup, but constructing fresh here is the canonical
# pattern most production code follows).

# YOUR CODE: Instantiate project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT,
# credential=cred).

# YOUR CODE: Exercise one cheap read so the credential actually resolves.
# Iterate project_client.connections.list() and collect the names into a list.
# Store the list in connection_names.

print(f"AIProjectClient authenticated. Found {len(connection_names)} connection(s).")
for name in connection_names:
    print(f"  - {name}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: AIProjectClient is instantiated with a DefaultAzureCredential
# (not AzureKeyCredential) and a low-cost project-level method returns
# without raising ClientAuthenticationError or HttpResponseError.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        if "project_client" not in globals():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "project_client is not defined in this notebook session. "
                    "Construct an AIProjectClient in Task 3 first."
                ),
            )

        credential = getattr(project_client, "_credential", None) or getattr(project_client, "credential", None)
        if credential is None:
            return CheckpointResult(
                status="fail",
                error_reason="AIProjectClient has no credential attribute exposed; check construction.",
            )
        cred_type = type(credential).__name__
        if "Key" in cred_type and "Credential" in cred_type:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AIProjectClient was constructed with {cred_type!r}, which is an "
                    f"API-key credential. Production code must use DefaultAzureCredential. "
                    f"Re-run Task 3 with credential=DefaultAzureCredential()."
                ),
            )
        if cred_type not in (
            "DefaultAzureCredential",
            "ChainedTokenCredential",
            "ManagedIdentityCredential",
            "AzureCliCredential",
            "EnvironmentCredential",
        ):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Credential type is {cred_type!r}. Expected DefaultAzureCredential or one of "
                    f"its inner credentials."
                ),
            )

        try:
            list(project_client.connections.list())
        except ClientAuthenticationError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"connections.list() raised ClientAuthenticationError: {e.message}. "
                    f"Your principal needs Azure AI Developer on the resource group."
                ),
            )
        except HttpResponseError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"connections.list() raised HttpResponseError {e.status_code}: {e.message}. "
                    f"Check that PROJECT_ENDPOINT points at the project (not the hub)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint inspects the credential attached to the client and rejects key-based credentials. It also runs `connections.list()` itself; if your client constructs but the call returns a 401 or 403, your principal is missing the `Azure AI Developer` role on the resource group.

</details>

<details><summary>Hint 2 (direction)</summary>

Two lines. `cred = DefaultAzureCredential()` then `project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cred)`. The endpoint must be the project's `discovery_url` (the project workspace `properties.discovery_url`), not the hub's. Hub endpoints return 404 from `connections.list()`. The third line exercises the credential: `connection_names = [c.name for c in project_client.connections.list()]`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
cred = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cred)
connection_names = [c.name for c in project_client.connections.list()]
```

</details>

**Wallet check.** Still at $0.00. Listing connections is a control-plane read and free. The token meter has not started. Coffee is still ahead.

## Task 4: Run a five-turn chat and read usage metadata from every response

Production chat apps need to know what each call cost in tokens so the team can capacity-plan deployments and surface per-user budgets. The `usage` field on every chat-completion response carries `prompt_tokens` and `completion_tokens`. Walking that field is muscle memory the exam tests.

Build it in this order:

1. Get the chat-completions client from the project: `chat = project_client.inference.get_chat_completions_client()`.
2. Maintain a `messages` list seeded with a system prompt. For five turns:
   - Append a user message.
   - Call `chat.complete(model=DEPLOYMENT_NAME, messages=messages)`.
   - Append the assistant message (so the NEXT turn sees prior history).
   - Capture `response.usage.prompt_tokens` and `response.usage.completion_tokens` into `USAGE_LOG`.
3. After the loop, compute the total cost at $0.15/1M input + $0.60/1M output and confirm it stays under two cents.

Trap to avoid: a common mistake is calling `chat.complete(...)` with a single-turn `messages=[{user message}]` every iteration. That makes `prompt_tokens` flat across turns; the checkpoint catches it and tells you to thread prior history. The whole point of the `messages` list is to grow it each turn.

In [ ]:
# NBVAL_SKIP
# Task 4: Five-turn chat completion. Each turn appends the user message and
# the assistant reply to the messages list so prompt_tokens grows turn over
# turn. Captured usage is written to USAGE_LOG which the checkpoint reads.

chat = project_client.inference.get_chat_completions_client()

system_prompt = (
    "You are a concise assistant. Answer each question in under 60 words. "
    "Build on the prior conversation context."
)
user_turns = [
    "What is the relationship between a Foundry hub and a Foundry project?",
    "Which one would I attach an Azure OpenAI deployment to in this lab?",
    "If I want three apps sharing the same models, do I make three hubs?",
    "Why is DefaultAzureCredential the recommended auth path for production?",
    "Summarise our conversation in one sentence.",
]

messages = [{"role": "system", "content": system_prompt}]
USAGE_LOG = []

for turn_index, user_message in enumerate(user_turns, start=1):
    messages.append({"role": "user", "content": user_message})
    # YOUR CODE: Call response = chat.complete(model=DEPLOYMENT_NAME, messages=messages)
    # so the model can see every prior turn.

    assistant_content = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_content})

    # YOUR CODE: Capture usage. Append a dict
    # {"turn": turn_index, "prompt_tokens": response.usage.prompt_tokens,
    # "completion_tokens": response.usage.completion_tokens} to USAGE_LOG.

    print(
        f"Turn {turn_index}: prompt_tokens={response.usage.prompt_tokens}, "
        f"completion_tokens={response.usage.completion_tokens}"
    )

total_input = sum(entry["prompt_tokens"] for entry in USAGE_LOG)
total_output = sum(entry["completion_tokens"] for entry in USAGE_LOG)
total_cost = (total_input / 1_000_000.0) * PRICE_PER_1M_INPUT_USD + (total_output / 1_000_000.0) * PRICE_PER_1M_OUTPUT_USD

print()
print(f"Five turns complete. Input tokens: {total_input}. Output tokens: {total_output}.")
print(f"Total session cost: ${total_cost:.6f} (ceiling ${COST_CEILING_USD:.2f}).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Five completed turns, each with non-zero prompt and
# completion token counts, prompt_tokens rising across turns (multi-turn
# context threaded correctly), and the cumulative cost under two cents.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        if not isinstance(USAGE_LOG, list) or len(USAGE_LOG) < 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"USAGE_LOG has {len(USAGE_LOG) if isinstance(USAGE_LOG, list) else 'no'} "
                    f"entries, expected 5. Did all five turns of Task 4 complete?"
                ),
            )

        prompt_tokens = []
        for i, entry in enumerate(USAGE_LOG[:5], start=1):
            pt = entry.get("prompt_tokens", 0)
            ct = entry.get("completion_tokens", 0)
            if not pt or pt <= 0:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Turn {i} prompt_tokens is {pt}; expected a positive integer.",
                )
            if not ct or ct <= 0:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Turn {i} completion_tokens is {ct}; the assistant returned an empty "
                        f"reply or the response.usage field was misread."
                    ),
                )
            prompt_tokens.append(pt)

        # prompt_tokens should rise across turns because each turn appends prior
        # context. Flat or near-flat counts mean the messages list was reset.
        if prompt_tokens[-1] <= prompt_tokens[0]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"prompt_tokens did not rise across turns (turn 1: {prompt_tokens[0]}, "
                    f"turn 5: {prompt_tokens[-1]}). Each turn should append both the user "
                    f"message and the assistant reply to messages so the next turn carries history."
                ),
            )

        total_input = sum(prompt_tokens)
        total_output = sum(entry["completion_tokens"] for entry in USAGE_LOG[:5])
        total_cost = (total_input / 1_000_000.0) * PRICE_PER_1M_INPUT_USD + (
            total_output / 1_000_000.0
        ) * PRICE_PER_1M_OUTPUT_USD
        if total_cost > COST_CEILING_USD:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Total session cost ${total_cost:.6f} exceeds the ${COST_CEILING_USD:.2f} "
                    f"ceiling. Shorten the system prompt or the user turns."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint says exactly which assertion failed. If `prompt_tokens did not rise`, the messages list is being reset somewhere in the loop. If a turn's `completion_tokens` is zero, the model returned an empty content (the response was probably filtered or rate-limited). If the cost ceiling fires, your system prompt is too long.

</details>

<details><summary>Hint 2 (direction)</summary>

Two things must happen inside the loop: call `chat.complete(model=DEPLOYMENT_NAME, messages=messages)` (the full growing list, not the single turn), and after capturing `assistant_content`, append `{"role": "assistant", "content": assistant_content}` BEFORE the next iteration. Usage is read from `response.usage.prompt_tokens` and `response.usage.completion_tokens` and appended as a dict to `USAGE_LOG`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the body of the loop):

```python
response = chat.complete(model=DEPLOYMENT_NAME, messages=messages)

USAGE_LOG.append({
    "turn": turn_index,
    "prompt_tokens": response.usage.prompt_tokens,
    "completion_tokens": response.usage.completion_tokens,
})
```

</details>

**Wallet check.** About a fifth of a penny. Five turns at roughly 200 to 500 input tokens and 100 to 200 output tokens each lands around 2,500 input and 1,000 output tokens for the session, which at $0.15 per 1M input and $0.60 per 1M output works out to roughly $0.0004 plus $0.0006 = about $0.001 total. Your morning coffee is somewhere around 5,000x more expensive.

## Cleanup

Time to tear it all down. The cell below runs through your manifest in reverse-creation order (deployment first, then project, then hub, then resource group), deletes every resource, then double-checks Azure Resource Graph to confirm nothing tagged with this lab's slug is left. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 01 has no critical (hourly-billed) resources, so the canonical
# summary always reports zero critical destructions.

import sys

# Update the AOAI deployment cli_delete_command now that AOAI_ACCOUNT_NAME is
# resolved (the manifest cell ran before discovery in Task 2).
for r in CLEANUP_MANIFEST:
    if r.type == "aoai_deployment" and AOAI_ACCOUNT_NAME and "<attached-aoai-account>" in (r.cli_delete_command or ""):
        r.cli_delete_command = r.cli_delete_command.replace(
            "<attached-aoai-account>", AOAI_ACCOUNT_NAME,
        )

# Patch resource.extra with constants resolved during the lab so the
# Azure adapter sees account_name, service_name, and project_endpoint
# at cleanup time (manifest is built before those constants are set).
for r in CLEANUP_MANIFEST:
    if r.type in ("aoai_deployment", "aoai_rai_policy") and AOAI_ACCOUNT_NAME:
        r.extra = {"account_name": AOAI_ACCOUNT_NAME}
    elif r.type == "search_index" and "SEARCH_SERVICE_NAME" in globals() and SEARCH_SERVICE_NAME:
        r.extra = {"service_name": SEARCH_SERVICE_NAME}
    elif r.type == "foundry_agent" and "PROJECT_ENDPOINT" in globals() and PROJECT_ENDPOINT:
        r.extra = {"project_endpoint": PROJECT_ENDPOINT}

result = run_cleanup(CLEANUP_MANIFEST, adapter=AzureCleanupAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Azure subscription may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about a fifth of a penny.** Five GPT-4o-mini calls at the published $0.15 / $0.60 per 1M token rates barely move the meter. The Foundry hub, project, and Standard deployment carry no hourly fee at zero traffic, so the resource group delete in cleanup is the safety net, not the cost-stopper. Check your Azure Cost Management view in 24 hours to confirm the exact number (Cost Management has a 24-hour-plus data lag, same as AWS Cost Explorer).

## Reflection

These are not graded. They are for you.

1. Walk through what a Foundry hub gives you that a bare Azure OpenAI account does not. What does the project layer add on top of the hub? Why would a team with three apps building against the same models still create three projects under one hub instead of one hub per app?

2. Your team has a production Foundry deployment authenticated via an API key kept in an Azure Key Vault. The security review wants to move to managed identity in the next sprint. What changes in your application code, what changes in your deployment infrastructure, and what changes in your incident-response playbook when the credential is no longer a string you can rotate by hand?

## Exam-Style Practice

**Question 1 (Domain 1, model selection):**

You are building a customer-support chat app that runs about 200,000 conversational turns per day, each averaging 300 input tokens and 100 output tokens. The product owner wants the lowest viable Azure OpenAI cost without sacrificing chat-completion quality. Which deployment configuration fits?

A. `gpt-4o` Standard deployment in `eastus` at $2.50 / $10.00 per 1M tokens.

B. `gpt-4o-mini` Standard deployment in `eastus` at $0.15 / $0.60 per 1M tokens.

C. `gpt-4.1` Provisioned-throughput deployment with 100 PTUs reserved monthly.

D. `gpt-5-nano` Standard deployment, accepting that the model card lists chat quality as below `gpt-4o-mini`.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: `gpt-4o` is roughly 16x more expensive on input and output than `gpt-4o-mini` with marginal quality gain for routine chat support. Cost-inefficient for this workload.
- B is correct: `gpt-4o-mini` at $0.15 / $0.60 per 1M is Microsoft's published recommended default for chat workloads and matches the AI-103 study guide's example pricing. 200,000 turns x (300 input + 100 output) tokens = 60M input + 20M output = $9 + $12 = $21/day. Lowest viable cost with chat-grade quality.
- C is wrong: Provisioned-throughput is the wrong pricing model for predictable-but-modest workloads. PTU is committed hourly capacity, charged whether used or not. This workload does not justify the up-front commit.
- D is wrong: while `gpt-5-nano` is cheaper at $0.05 / $0.40 per 1M, the model card explicitly notes lower chat quality vs `gpt-4o-mini`. The question requires no quality sacrifice.

</details>

**Question 2 (Domain 1, keyless auth and security):**

A Foundry developer is shipping a chat app to production and security requires that no API keys leave the secret manager. The app runs in an Azure Container App. Which credential approach satisfies the requirement with the minimum code change?

A. Store the Foundry account API key in Azure Key Vault, read it at container startup, and pass it to `AzureKeyCredential`.

B. Assign a system-assigned managed identity to the Container App, grant it `Cognitive Services OpenAI User` on the Azure OpenAI resource, and use `DefaultAzureCredential` in code.

C. Use a Microsoft Entra ID user account, sign in with `az login` in container startup, and rely on the cached token.

D. Disable authentication on the Foundry endpoint and restrict via VNet integration only.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: the API key still leaves Key Vault and lives in process memory; rotation requires app restart; security can object to keys-in-process even when sourced from a vault.
- B is correct: system-assigned managed identity gives the Container App an Entra ID identity tied to its lifecycle. `DefaultAzureCredential` resolves it transparently. No secret material exists; the identity rotates on container delete. Minimum code change is the `DefaultAzureCredential()` constructor.
- C is wrong: `az login` is interactive and inappropriate for unattended container startup.
- D is wrong: disabling auth is a non-starter on a production endpoint. VNet integration is a network control, not an authentication control.

</details>